# ️ Glu-Stock: 01_RESEARCH_SCAN
**Phase**: Universe Selection & Fundamental Filtering

This notebook scans the IDX market, filters for liquidity and high-growth fundamentals, and pushes candidates to the Firebase `research` queue.

In [ ]:
!pip install -q yfinance firebase-admin pandas python-dotenv ta


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Universe)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                print(f"[ERROR] FIREBASE_KEY_JSON missing or invalid! Error: {e}")
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_full_idx_universe():
    # Standard mainboard (Papan Utama) fallback list (259 tickers)
    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    return fallback


In [ ]:
#  SECTION 3: CORE LOGIC (High-Efficiency Quant Scanner)
class QuantScanner:
    @staticmethod
    def run_scan(universe):
        print(f'[SCAN] Batch downloading market data for {len(universe)} stocks...')
        # Download 1 year (252 days) for SMA 200 calculation
        data = yf.download(universe, period='1y', interval='1d', progress=True, auto_adjust=True)
        
        candidates = []
        n_ok, n_trend, n_liq, n_err = 0, 0, 0, 0
        
        for ticker in universe:
            try:
                # 1. Extract columns safety
                close = data['Close'][ticker].dropna()
                volume = data['Volume'][ticker].dropna()
                
                if len(close) < 200:
                    continue
                
                # 2. Stage 1: Liquidity Filter (Avg Volume > 500k shares)
                avg_vol = volume.tail(20).mean()
                if avg_vol < 500000:
                    n_liq += 1
                    continue
                
                # 3. Stage 2: Trend Filter (Price > SMA 200)
                sma200 = close.mean() # Close is ~252 samples
                curr_price = close.iloc[-1]
                if curr_price < sma200:
                    n_trend += 1
                    continue
                
                # 4. Stage 3: Momentum Ranking (3-Month Returns)
                ret_3m = (close.iloc[-1] / close.iloc[-60]) - 1
                
                candidates.append({
                    'ticker': ticker,
                    'price': round(curr_price, 2),
                    'momentum_3m': round(ret_3m, 4),
                    'avg_vol': int(avg_vol)
                })
                n_ok += 1
                
            except Exception as e:
                n_err += 1
                continue
        
        # Sort by Momentum Score
        candidates = sorted(candidates, key=lambda x: x['momentum_3m'], reverse=True)
        
        print(f'[REPORT] Scan Report: OK={n_ok} | Weak Trend={n_trend} | Low Liquidity={n_liq} | Error={n_err}')
        return candidates


In [ ]:
#  SECTION 4: MAIN EXECUTION
def run_workflow():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    universe = get_full_idx_universe()
    candidates = QuantScanner.run_scan(universe)
    
    if candidates:
        # Push only the ticker list for the inference queue
        ticker_list = [c['ticker'] for c in candidates]
        fb.push_task("research", ticker_list)
        
        msg = f"Scan Complete: {len(ticker_list)} candidates found.\nTop 3: {ticker_list[:3]}"
        fb.log_event("RESEARCH", msg)
        print(f"[OK] {msg}")
    else:
        print("[EMPTY] No candidates met the institutional criteria today.")

run_workflow()
